# 自建 VPS 教程

>登陆web:  https://my.vultr.com/
>选择 Cloud Compute,创建vps


## 服务器配置
## ssh 连接 remote VPS


In [ ]:
%%bash
#!/bin/bash
# 用法: ./ss_deploy.sh <服务器IP> <密码> [配置名]
# 示例: ./ss_deploy.sh 202.182.112.91 mypassword tokyo

IP="${1:?需要服务器IP}"
PASSWORD="${2:?需要密码}"
CONFIG="${3:-config}"
SCRIPT=/data/.manjaro/utils/ss_server_conf.sh

# 1. 建立免密 SSH 认证
echo "🔑 配置免密登录..."
[ ! -f ~/.ssh/id_ed25519 ] && ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519
ssh-keyscan -H "$IP" >> ~/.ssh/known_hosts 2>/dev/null
ssh-copy-id -i ~/.ssh/id_ed25519.pub root@"$IP"

# 2. 上传并执行配置脚本
echo "📤 上传配置脚本..."
scp "$SCRIPT" root@"$IP":~/ss_server_conf.sh

echo "🚀 远程执行配置..."
ssh root@"$IP" "bash ss_server_conf.sh '$PASSWORD' 8388 aes-256-gcm '$CONFIG'"


## 本地配置


### ssh


In [ ]:
ssh -D 1080 -N -f -C root@202.182.112.91


### clash


In [ ]:
%cd /data/.manjaro
from utils.clash import generate

ip = '202.182.112.91'
ss_config = {
    "server": "0.0.0.0",
    "server_port": 8388,
    "password": "lkf.Vpn.mima3",
    "method": "aes-256-gcm",
}

generate(ip, ss_config, node="Tokyo", out="/tmp/cash.yaml")


### shadowsocks

In [7]:
%%bash
#!/bin/bash
# 用法: ./ss_local.sh [服务器IP] [密码]

IP="${1:-202.182.112.91}"
PASSWORD="${2:-lkf.Vpn.mima3}"
cfg=$HOME/.shadowsocks/config.json
pid=$HOME/.shadowsocks/ss.pid
log=$HOME/.shadowsocks/ss.log

sudo pacman -S --needed --noconfirm shadowsocks-rust

mkdir -p $(dirname $cfg)
cat > $cfg << EOF
{
    "server": "$IP",
    "server_port": 8388,
    "local_address": "127.0.0.1",
    "local_port": 1080,
    "password": "$PASSWORD",
    "method": "aes-256-gcm",
    "timeout": 300,
    "fast_open": false
}
EOF

pkill -f sslocal 2>/dev/null
sslocal -c $cfg -d start --pid-file $pid --log-file $log
sleep 1
echo "sslocal pid: $(cat $pid 2>/dev/null)"


 there is nothing to do


error: unexpected argument 'start' found

Usage: sslocal [OPTIONS]

For more information, try '--help'.


sslocal pid: 387512
